# Comprehensive Emergency Triage Modeling Pipeline

This notebook provides an end-to-end pipeline for predicting emergency triage acuity (ESI levels) using the NHAMCS dataset (2018-2022). It integrates clinical tabular data with NLP-based chief complaint analysis using a multi-stage modeling approach.

## Pipeline Architecture
1. **Data Loading & Preprocessing**: Loading NHAMCS data and grouping ESI levels into three clinical categories.
2. **Feature Engineering**:
   - **Tabular**: Cyclical time encoding, clinical ratios (Shock Index, NEWS2), and vital missingness.
   - **Text**: Emergency keyword flagging with negation handling.
3. **Stage 1: NLP Model**: Fine-tuning DistilBERT with a CORN (Conditional Ordinal Regression) head on chief complaints.
4. **Stage 2: Tree-based Models**: Training XGBoost and LightGBM (Classifiers & Regressors) + Balanced Random Forest on tabular features.
5. **Stage 3: Meta-Learner**: A Stacked Logistic Regression model combining base model predictions.
6. **Evaluation**: Comprehensive assessment using Quadratic Weighted Kappa (QWK) and classification reports.

In [ ]:
import os
import re
import json
import random
import warnings
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import joblib
import xgboost as xgb
import lightgbm as lgb

from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.model_selection import GroupKFold, train_test_split
from sklearn.metrics import classification_report, cohen_kappa_score, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from sklearn.linear_model import LogisticRegression
from sklearn.base import clone
from imblearn.ensemble import BalancedRandomForestClassifier
from scipy.optimize import minimize
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

# Constants & Seeds
SEED = 42
RANDOM_STATE = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_NAME = "nlpie/distil-clinicalbert"
NUM_CLASSES = 3
MAX_LEN = 100
TRAIN_BATCH_SIZE = 32
EVAL_BATCH_SIZE = 16
LR = 2e-5
EPOCHS = 2

PROJECT_ROOT = Path("..").resolve()
DATA_PATH = PROJECT_ROOT / "working_data" / "nhamcs_data_2018_22.csv"
RESULTS_DIR = PROJECT_ROOT / "results" / "final_submission"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Device: {DEVICE}")
print(f"Data path exists: {DATA_PATH.exists()}")

## 1. Data Loading & Category Mapping

The Emergency Severity Index (ESI) is mapped from 1-5 into three clinical categories:
- **Category 0 (Urgent)**: ESI 1 and 2
- **Category 1 (Emergent)**: ESI 3
- **Category 2 (Non-Urgent)**: ESI 4 and 5

In [ ]:
def map_esi_to_group(val):
    if val in (1, 2): return 0
    if val == 3:      return 1
    if val in (4, 5): return 2
    return np.nan

df = pd.read_csv(DATA_PATH)
df.columns = df.columns.astype(str).str.strip()

print(f"Raw data loaded: {df.shape}")

target_col = "target_triage_acuity"
if target_col in df.columns:
    df = df.dropna(subset=[target_col]).copy()
    df['target_group'] = df[target_col].map(map_esi_to_group)
    df = df.dropna(subset=['target_group']).copy()
    df['target_group'] = df['target_group'].astype(int)

print(f"Processed rows: {len(df):,}")
print("Target Distribution:")
print(df['target_group'].value_counts().sort_index())

## 2. Tabular Feature Engineering

We implement cyclical encoding for time-based features and derive clinical indicators like the NEWS2 score and Shock Index.

In [ ]:
print("Applying Tabular Feature Engineering...")

# 1. Cyclical Encoding for Time Features
if 'arrival_time' in df.columns:
    df['arrival_time'] = np.where(df['arrival_time'] < 0, np.nan, df['arrival_time'])
    hours = np.floor_divide(df['arrival_time'], 100)
    minutes = np.mod(df['arrival_time'], 100)
    df['arrival_hour_float'] = hours + (minutes / 60.0)

cycle_maxes = {'visit_month': 12.0, 'day_of_week': 7.0, 'arrival_hour_float': 24.0}
for col, max_val in cycle_maxes.items():
    if col in df.columns:
        angle = 2 * np.pi * df[col] / max_val
        df[f'{col}_sin'] = np.sin(angle)
        df[f'{col}_cos'] = np.cos(angle)

# 2. Clinical Ratios & Indicators
if "heart_rate" in df.columns and "sys_bp" in df.columns:
    df["shock_index"] = df["heart_rate"] / df["sys_bp"].replace(0, np.nan)
    
if "sys_bp" in df.columns and "dias_bp" in df.columns:
    df["map"] = (df["sys_bp"] + 2 * df["dias_bp"]) / 3
    df["pulse_pressure"] = df["sys_bp"] - df["dias_bp"]

if "resp_rate" in df.columns and "spo2" in df.columns and "sys_bp" in df.columns:
    news2 = pd.Series(0, index=df.index, dtype=float)
    rr = df["resp_rate"]
    news2 += np.select([rr <= 8, (rr>=9)&(rr<=11), (rr>=12)&(rr<=20), (rr>=21)&(rr<=24), rr>=25], [3, 1, 0, 2, 3], default=0)
    sp = df["spo2"]
    news2 += np.select([sp<=91, (sp>=92)&(sp<=93), (sp>=94)&(sp<=95), sp>=96], [3, 2, 1, 0], default=0)
    sbp = df["sys_bp"]
    news2 += np.select([sbp<=90, (sbp>=91)&(sbp<=100), (sbp>=101)&(sbp<=110), (sbp>=111)&(sbp<=219), sbp>=220], [3, 2, 1, 0, 3], default=0)
    df["news2_score"] = news2

vital_cols = ["heart_rate", "sys_bp", "dias_bp", "resp_rate", "spo2", "temp"]
for col in vital_cols:
    if col in df.columns:
        df[f"{col}_missing"] = df[col].isna().astype(int)

num_cols = df.select_dtypes(include=["number"]).columns
df[num_cols] = df[num_cols].replace([np.inf, -np.inf], np.nan).fillna(df[num_cols].median())

print(f"Tabular FE complete. New shape: {df.shape}")

## 3. Text Preprocessing & Emergency Flagging

We extract clinical red flags from the chief complaint and injury cause text using strict negation handling.

In [ ]:
ABBREV_MAP = {"sob":"shortness of breath","cp":"chest pain","loc":"loss of consciousness","ams":"altered mental status", "sz":"seizure", "mi":"myocardial infarction", "oth":"other", "usp": "unspecified"}
EMERGENCY_KEYWORDS = {
    "chest_pain": "chest pain", 
    "shortness_of_breath": "shortness of breath",
    "syncope": "syncope",
    "assault": "assault",
    "vaginal_bleeding": "vaginal bleeding",
    "violence": "violence",
    "burn": "burn",
    "head_injury": "head injury",
    "suicide_attempt": "suicide attempt",
    "cardiac_arrest": "cardiac arrest",
    "gunshot_wound": "gunshot wound",
    "sepsis": "sepsis"
}
NEGATION_TOKENS = {"no", "not", "denies", "denied", "without", "negative", "absent", "never"}

def normalize_text(text):
    if pd.isna(text): return ""
    t = str(text).lower().strip()
    t = t.replace("_", " ")
    t = re.sub(r"[^a-z0-9\s]", " ", t)
    for abbr, expanded in ABBREV_MAP.items():
        t = re.sub(rf"\b{abbr}\b", expanded, t)
    return re.sub(r"\s+", " ", t).strip()

def is_negated(text, keyword):
    pattern = re.compile(rf"(?<!\w){re.escape(keyword)}(?!\w)")
    match = pattern.search(text)
    if not match: return False
    left_context = text[max(0, match.start() - 40):match.start()]
    tokens = re.findall(r"\w+", left_context)
    return any(tok in NEGATION_TOKENS for tok in tokens[-5:])

df["proc_chief_complaint"] = df["chief_complaint_text"].map(normalize_text)
df["proc_injury_cause"] = df["injury_cause_text"].map(normalize_text)
df["combined_text"] = df["proc_chief_complaint"] + " " + df["proc_injury_cause"]

for flag_name, keyword in EMERGENCY_KEYWORDS.items():
    df[f"flag_{flag_name}"] = df["combined_text"].apply(lambda x: 1 if (keyword in x and not is_negated(x, keyword)) else 0)

print("Emergency flags added.")

## 4. Stage 1: Deep Learning (NLP) Model

We train a clinical transformers model using the CORN (Conditional Ordinal Regression) framework to predict triage category from chief complaints.

In [ ]:
class OrdinalTextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=96):
        self.texts = texts.astype(str).tolist()
        self.labels = labels.astype(int).tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], truncation=True, padding="max_length", max_length=self.max_len, return_tensors="pt")
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["label"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

class OrdinalBert(nn.Module):
    def __init__(self, model_name, num_classes=3, dropout=0.5):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name, use_safetensors=True)
        hidden = self.backbone.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.ordinal_head = nn.Linear(hidden, num_classes - 1)

    def forward(self, input_ids, attention_mask):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        cls_state = out.last_hidden_state[:, 0]
        return self.ordinal_head(self.dropout(cls_state))

def corn_loss(logits, y_idx, num_classes=3):
    total_loss = torch.tensor(0.0, device=logits.device)
    total_count = 0
    for k in range(num_classes - 1):
        mask = torch.ones_like(y_idx, dtype=torch.bool) if k == 0 else y_idx >= k
        if mask.sum() == 0: continue
        target = (y_idx[mask] > k).float()
        loss_k = F.binary_cross_entropy_with_logits(logits[mask, k], target, reduction="sum")
        total_loss += loss_k
        total_count += int(mask.sum().item())
    return total_loss / max(1, total_count)

def logits_to_class_probs(logits_tensor):
    cond_probs = torch.sigmoid(logits_tensor)
    cum_probs = torch.cumprod(cond_probs, dim=1)
    probs = torch.zeros((cum_probs.size(0), NUM_CLASSES), dtype=cum_probs.dtype, device=cum_probs.device)
    probs[:, 0] = 1.0 - cum_probs[:, 0]
    for c in range(1, NUM_CLASSES - 1):
        probs[:, c] = cum_probs[:, c - 1] - cum_probs[:, c]
    probs[:, NUM_CLASSES - 1] = cum_probs[:, NUM_CLASSES - 2]
    return torch.clamp(probs, min=0.0)

def run_epoch(model, loader, optimizer=None, scheduler=None):
    is_train = optimizer is not None
    model.train(is_train)
    all_logits = []
    with (torch.enable_grad() if is_train else torch.no_grad()):
        for batch in loader:
            ids, mask, y = batch["input_ids"].to(DEVICE), batch["attention_mask"].to(DEVICE), batch["label"].to(DEVICE)
            logits = model(ids, mask)
            if is_train:
                loss = corn_loss(logits, y)
                loss.backward(); optimizer.step(); scheduler.step(); optimizer.zero_grad()
            all_logits.append(logits.detach().cpu())
    return torch.cat(all_logits, dim=0)

In [ ]:
print("Starting Stage 1: NLP Model Training (3-Fold CV)")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
year_groups = df["year"]
years_sorted = np.sort(year_groups.unique())
year_bins = np.array_split(years_sorted, 3)
year_bucket_map = {y: idx for idx, years in enumerate(year_bins) for y in years}
year_bucket = year_groups.map(year_bucket_map)
gkf = GroupKFold(n_splits=3)

oof_nlp_probs = np.zeros((len(df), NUM_CLASSES))

for fold, (tr_idx, val_idx) in enumerate(gkf.split(df, df['target_group'], groups=year_bucket), 1):
    print(f"Fold {fold}...")
    tr_ds = OrdinalTextDataset(df.iloc[tr_idx]["proc_chief_complaint"], df.iloc[tr_idx]["target_group"], tokenizer)
    val_ds = OrdinalTextDataset(df.iloc[val_idx]["proc_chief_complaint"], df.iloc[val_idx]["target_group"], tokenizer)
    
    tr_loader = DataLoader(tr_ds, batch_size=TRAIN_BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=EVAL_BATCH_SIZE, shuffle=False)
    
    model = OrdinalBert(MODEL_NAME).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
    scheduler = get_linear_schedule_with_warmup(optimizer, 0, len(tr_loader) * EPOCHS)
    
    for epoch in range(EPOCHS):
        run_epoch(model, tr_loader, optimizer, scheduler)
    
    val_logits = run_epoch(model, val_loader)
    oof_nlp_probs[val_idx] = logits_to_class_probs(val_logits).numpy()

print("NLP OOF complete.")

## 5. Stage 2: Tree-Based Models (Classifiers & Regressors)

We train XGBoost and LightGBM models in both classification and regression modes, alongside Balanced Random Forest. Regressors are calibrated using custom cutpoints to optimize for ordinal performance.

In [ ]:
def fit_cutpoints(y_true, y_pred, n_cls):
    df_cut = pd.DataFrame({"y": y_true, "pred": y_pred})
    medians = df_cut.groupby("y")["pred"].median().sort_index()
    if len(medians) < n_cls: return np.quantile(y_pred, np.linspace(0, 1, n_cls + 1)[1:-1])
    return np.array([(medians.iloc[i] + medians.iloc[i + 1]) / 2.0 for i in range(n_cls - 1)])

def regressor_soft_proba(train_pred, val_pred, y_train, n_cls):
    centers = pd.Series(train_pred).groupby(y_train.values).median()
    if len(centers) < n_cls: centers = pd.Series(np.linspace(train_pred.min(), train_pred.max(), n_cls), index=range(n_cls))
    centers = centers.reindex(range(n_cls)).interpolate().to_numpy()
    tau = np.var(train_pred) + 1e-6
    dist = (val_pred[:, None] - centers[None, :]) ** 2
    probs = np.exp(-dist / (2.0 * tau))
    return probs / probs.sum(axis=1, keepdims=True)

In [ ]:
print("Starting Stage 2: Tree-Based Model Training (3-Fold CV)")

cols_to_drop = ["target_triage_acuity", "target_group", "year", "visit_month", "day_of_week", "arrival_time", 
                "chief_complaint_text", "injury_cause_text", "proc_chief_complaint", "proc_injury_cause", "combined_text"]
X = df.drop(columns=[c for c in cols_to_drop if c in df.columns]).copy()
for col in X.select_dtypes(include=["object", "string", "category"]).columns:
    X[col] = X[col].astype("category").cat.codes
y = df["target_group"]

base_oof = {"xgb_cls": np.zeros((len(df), NUM_CLASSES)), "lgb_cls": np.zeros((len(df), NUM_CLASSES)),
            "xgb_reg": np.zeros((len(df), NUM_CLASSES)), "lgb_reg": np.zeros((len(df), NUM_CLASSES)),
            "brf": np.zeros((len(df), NUM_CLASSES))}

models = {
    "xgb_cls": xgb.XGBClassifier(n_estimators=400, max_depth=6, learning_rate=0.05, enable_categorical=True, random_state=SEED),
    "lgb_cls": lgb.LGBMClassifier(n_estimators=400, learning_rate=0.05, verbose=-1, random_state=SEED),
    "xgb_reg": xgb.XGBRegressor(n_estimators=400, max_depth=6, learning_rate=0.05, enable_categorical=True, random_state=SEED),
    "lgb_reg": lgb.LGBMRegressor(n_estimators=400, learning_rate=0.05, verbose=-1, random_state=SEED),
    "brf": BalancedRandomForestClassifier(n_estimators=300, random_state=SEED)
}

for fold, (tr_idx, val_idx) in enumerate(gkf.split(X, y, groups=year_bucket), 1):
    print(f"Fold {fold}...")
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
    
    for name, model in models.items():
        m = clone(model).fit(X_tr, y_tr)
        if "reg" in name:
            tr_preds, val_preds = m.predict(X_tr), m.predict(X_val)
            base_oof[name][val_idx] = regressor_soft_proba(tr_preds, val_preds, y_tr, NUM_CLASSES)
        else:
            base_oof[name][val_idx] = m.predict_proba(X_val)

print("Tree model OOF complete.")

## 6. Stage 3: Meta-Learner

The meta-learner combines the OOF predictions from all base models (Classifiers, Regressors, and NLP) using Stacked Logistic Regression.

In [ ]:
print("Starting Stage 3: Meta-Learner Training")

meta_X = np.hstack(list(base_oof.values()) + [oof_nlp_probs])
meta_oof_preds = np.zeros(len(df))

for fold, (tr_idx, val_idx) in enumerate(gkf.split(meta_X, y, groups=year_bucket), 1):
    X_mtr, X_mval = meta_X[tr_idx], meta_X[val_idx]
    y_mtr, y_mval = y.iloc[tr_idx], y.iloc[val_idx]
    
    lr = LogisticRegression(class_weight="balanced", random_state=SEED)
    lr.fit(X_mtr, y_mtr)
    meta_oof_preds[val_idx] = lr.predict(X_mval)

print("\n--- Final Meta-Learner Report ---")
print(classification_report(y, meta_oof_preds, target_names=["Urgent", "Emergent", "NonUrgent"]))
print(f"Final QWK: {cohen_kappa_score(y, meta_oof_preds, weights='quadratic'):.4f}")

## 7. Final Model Training & Export

We train the final models on the full dataset and save them for deployment.

In [ ]:
print("Training final models on full dataset...")

for name, model in models.items():
    m_final = clone(model).fit(X, y)
    joblib.dump(m_final, RESULTS_DIR / f"{name}_final.joblib")

final_lr = LogisticRegression(class_weight="balanced", random_state=SEED)
final_lr.fit(meta_X, y)
joblib.dump(final_lr, RESULTS_DIR / "meta_learner_final.joblib")

print(f"All final models saved to: {RESULTS_DIR}")